# Eicher Motors — Financial Performance Analysis
**Student:** Debjyoti Mukherjee | **Company:** Eicher Motors | **CA2 Project**

This notebook walks through cleaning, exploring, and modelling Eicher Motors' quarterly numbers, finishing with the two files the dashboard needs.


## 1. Bring in the data

In [2]:
from google.colab import files
uploaded = files.upload()  # pick eicher_financials.csv

import pandas as pd
raw = pd.read_csv("eicher_financials.csv")
raw.shape


ModuleNotFoundError: No module named 'google.colab'

## 2. Tidy up
Sort by period and patch any gaps using the neighbouring quarters' average.


In [3]:
raw['period_key'] = pd.to_datetime(raw['Period'], format='%b-%Y')
raw = raw.sort_values('period_key').drop(columns='period_key').reset_index(drop=True)

numeric_cols = raw.select_dtypes('number').columns
raw[numeric_cols] = raw[numeric_cols].interpolate(limit_direction='both')

raw.head()


NameError: name 'pd' is not defined

## 3. Label each quarter's profit direction

In [ ]:
def label_trend(change):
    return "Profit Grew" if change > 0 else "Profit Declined"

raw['Profit_Trend'] = raw['Net_Profit'].diff().apply(label_trend)
raw.loc[0, 'Profit_Trend'] = "Profit Grew"   # opening quarter, no prior comparison

raw[['Period', 'Net_Profit', 'Profit_Trend']]


## 4. Summary statistics and growth rates

In [ ]:
summary = raw[['Sales', 'Net_Profit', 'OPM_Percent']].describe().loc[['mean', '50%', 'std']]
summary.index = ['mean', 'median', 'std']
summary


In [ ]:
raw['Sales_Growth'] = raw['Sales'].pct_change() * 100
raw['Net_Profit_Growth'] = raw['Net_Profit'].pct_change() * 100
raw[['Period', 'Sales_Growth', 'Net_Profit_Growth']].round(2)


## 5. Visual check
Using Plotly here instead of static plots so the charts stay interactive inside the notebook itself.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=raw['Period'], y=raw['Sales'], mode='lines+markers', name='Sales'))
fig1.add_trace(go.Scatter(x=raw['Period'], y=raw['Net_Profit'], mode='lines+markers', name='Net Profit'))
fig1.update_layout(title="Eicher Motors: Sales vs Net Profit (₹ Cr)")
fig1.show()


In [ ]:
fig2 = px.bar(raw, x='Period', y='OPM_Percent', title="Operating Margin (%) Each Quarter",
              color='OPM_Percent', color_continuous_scale='Reds')
fig2.show()


In [ ]:
fig3 = px.box(raw, x='Profit_Trend', y='Net_Profit', color='Profit_Trend',
              title="Net Profit Spread: Growth Quarters vs Decline Quarters",
              color_discrete_map={'Profit Grew': 'firebrick', 'Profit Declined': 'gray'})
fig3.show()


## 6. Two quick models
Logistic Regression as the simple baseline, a Decision Tree as the non-linear alternative. Predicting `Profit_Trend` from growth and cost drivers.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

drivers = ['Sales_Growth', 'OPM_Percent', 'Interest', 'Other_Income']
model_data = raw.dropna(subset=drivers + ['Profit_Trend'])

X = model_data[drivers]
y = model_data['Profit_Trend']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

logit = LogisticRegression(max_iter=1000).fit(X_train, y_train)
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [accuracy_score(y_test, logit.predict(X_test)),
                 accuracy_score(y_test, tree.predict(X_test))]
})
results


**Caveat:** twelve quarterly rows is a very small sample for machine learning — treat the accuracy numbers as a demonstration of the workflow, not a dependable forecast.

## 7. Export for the dashboard

In [ ]:
raw.to_csv("eicher_financials_clean.csv", index=False)
results.to_csv("eicher_model_comparison.csv", index=False)
print("Saved eicher_financials_clean.csv and eicher_model_comparison.csv")
